In [3]:
import json

import yfinance as yf
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(base_url="https://api.groq.com/openai/v1")

In [4]:
def get_stock(ticker: str):
    stock = yf.Ticker(ticker)
    info = stock.info
    output = {
        "ticker": ticker,
        "company_name": info.get("shortName", ticker),
        "current_price": info.get("currentPrice", 0)
    }
    return json.dumps(output)

In [5]:
tools = [
    {
        "type": "function",
        "name": "get_stock",
        "description": "Retorna informações básicas de uma ação",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Símbolo da ação (ex: AAPL, NVDA)",
                },
            },
            "required": ["ticker"],
        },
    },
]

input_list = [{"role": "user", "content": "Qual o preço da ação da Apple?"}]

In [7]:
response = client.responses.parse(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    tools=tools,
    input=input_list,
)

In [8]:
response.model_dump()

{'id': 'resp_01kr2a14nceebrt5ywp3tddgt9',
 'created_at': 1778194092.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'meta-llama/llama-4-scout-17b-16e-instruct',
 'object': 'response',
 'output': [{'id': 'resp_01kr2a14nceecbzbwy9r80qbn0',
   'summary': [],
   'type': 'reasoning',
   'content': None,
   'encrypted_content': None,
   'status': 'completed'},
  {'arguments': '{"ticker":"AAPL"}',
   'call_id': '620nzmqvm',
   'name': 'get_stock',
   'type': 'function_call',
   'id': '620nzmqvm',
   'status': 'completed',
   'parsed_arguments': None}],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [{'name': 'get_stock',
   'parameters': {'properties': {'ticker': {'description': 'Símbolo da ação (ex: AAPL, NVDA)',
      'type': 'string'}},
    'required': ['ticker'],
    'type': 'object'},
   'strict': None,
   'type': 'function',
   'description': 'Retorna informações básicas de uma ação'}],
 'top_p': 1.0

In [12]:
for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = get_stock(**args)
        input_list.append(
            {
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": result,
            }
        )

In [15]:
input_list[1]["output"]

'{"ticker": "AAPL", "company_name": "Apple Inc.", "current_price": 287.44}'

In [ ]:
final_response = client.responses.parse(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    instructions="Responda com uma análise baseada nos dados retornados pela função.",
    tools=tools,
    input=input_list,
)

O preço atual da ação da Apple (AAPL) é de $287.44.


In [18]:
print(final_response.output_text)

O preço atual da ação da Apple (AAPL) é de $287.44.
